<a href="https://colab.research.google.com/github/Birnurdagli/Vize-Final/blob/main/MakineOgrenmesi_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Veri Setlerini Yukleme


In [ ]:
import pandas as pd

df_normal_loaded = pd.read_csv('/content/normal_radiomics.csv')

df_papilodem_loaded = pd.read_csv('/content/papilodem_radiomics.csv')

print("Normal Radiomics DataFrame İlk 5 Satırı:")
display(df_normal_loaded.head())

print("\nPapilödem Radiomics DataFrame İlk 5 Satırı:")
display(df_papilodem_loaded.head())

# Veri Setlerini Birleştirme ve Etiketleme



In [ ]:
import pandas as pd

df_normal = pd.read_csv('/content/normal_radiomics.csv')
df_papilodem = pd.read_csv('/content/papilodem_radiomics.csv')

df_normal['target'] = 0
df_papilodem['target'] = 1

df_combined = pd.concat([df_normal, df_papilodem], ignore_index=True)

print("Combined DataFrame Head:")
print(df_combined.head())
print("\nCombined DataFrame Shape:", df_combined.shape)

In [ ]:
print("\nTarget Variable Value Counts:")
print(df_combined['target'].value_counts())

# Hasta Bazında Veri Bölme



In [ ]:
groups = df_combined['PatientIndex']
y = df_combined['target']
X = df_combined.drop(columns=['target', 'PatientIndex', 'SideStandard'])

print(f"Shape of X: {X.shape}")
print(f"Shape of y: {y.shape}")
print(f"Shape of groups: {groups.shape}")
print(f"Number of unique patients: {groups.nunique()}")

# Veri Kümelerinin StratifiedGroupKFold ile Bölünmesi



In [ ]:
from sklearn.model_selection import StratifiedGroupKFold


sgk_test_split = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for train_val_idx, test_idx in sgk_test_split.split(X, y, groups):
    X_train_val, X_test = X.iloc[train_val_idx], X.iloc[test_idx]
    y_train_val, y_test = y.iloc[train_val_idx], y.iloc[test_idx]
    groups_train_val, groups_test = groups.iloc[train_val_idx], groups.iloc[test_idx]
    break


sgk_val_split = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)

for train_idx, val_idx in sgk_val_split.split(X_train_val, y_train_val, groups_train_val):
    X_train, X_val = X_train_val.iloc[train_idx], X_train_val.iloc[val_idx]
    y_train, y_val = y_train_val.iloc[train_idx], y_train_val.iloc[val_idx]
    groups_train, groups_val = groups_train_val.iloc[train_idx], groups_train_val.iloc[val_idx]
    break

print(f"Original Data Shape: {X.shape}")
print(f"X_train Shape: {X_train.shape}, y_train Shape: {y_train.shape}, Unique Patients in train: {groups_train.nunique()}")
print(f"X_val Shape: {X_val.shape}, y_val Shape: {y_val.shape}, Unique Patients in val: {groups_val.nunique()}")
print(f"X_test Shape: {X_test.shape}, y_test Shape: {y_test.shape}, Unique Patients in test: {groups_test.nunique()}")

print("\nClass distribution in y_train:")
print(y_train.value_counts(normalize=True))
print("\nClass distribution in y_val:")
print(y_val.value_counts(normalize=True))
print("\nClass distribution in y_test:")
print(y_test.value_counts(normalize=True))

# Veri Ön İşleme



In [ ]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_val = X_val.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

imputer = SimpleImputer(strategy='median')
X_train_imputed = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_val_imputed = pd.DataFrame(imputer.transform(X_val), columns=X_val.columns, index=X_val.index)
X_test_imputed = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns, index=X_test.index)

print("After Median Imputation:")
print(f"X_train_imputed Shape: {X_train_imputed.shape}")
print(f"X_val_imputed Shape: {X_val_imputed.shape}")
print(f"X_test_imputed Shape: {X_test_imputed.shape}")

print("\n--- Missing Value Controls ---")
print(f"NaNs in X_train_imputed: {X_train_imputed.isnull().sum().sum()}")
print(f"NaNs in X_val_imputed: {X_val_imputed.isnull().sum().sum()}")
print(f"NaNs in X_test_imputed: {X_test_imputed.isnull().sum().sum()}")

# Düşük Varyanslı Özelliklerin Kaldırılması (VarianceThreshold)





In [ ]:
from sklearn.feature_selection import VarianceThreshold

# 2. Düşük varyanslı özellikleri kaldırın.
selector = VarianceThreshold(threshold=0.01)

# Fit on X_train_imputed and transform all sets
X_train_low_variance_removed = pd.DataFrame(selector.fit_transform(X_train_imputed), columns=X_train_imputed.columns[selector.get_support()], index=X_train_imputed.index)
X_val_low_variance_removed = pd.DataFrame(selector.transform(X_val_imputed), columns=X_val_imputed.columns[selector.get_support()], index=X_val_imputed.index)
X_test_low_variance_removed = pd.DataFrame(selector.transform(X_test_imputed), columns=X_test_imputed.columns[selector.get_support()], index=X_test_imputed.index)

print("\nAfter Low Variance Feature Removal:")
print(f"X_train_low_variance_removed Shape: {X_train_low_variance_removed.shape}")
print(f"X_val_low_variance_removed Shape: {X_val_low_variance_removed.shape}")
print(f"X_test_low_variance_removed Shape: {X_test_low_variance_removed.shape}")

print(f"Number of features removed: {X_train_imputed.shape[1] - X_train_low_variance_removed.shape[1]}")

In [ ]:
import numpy as np
corr_matrix = X_train_low_variance_removed.corr().abs()

upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

to_drop = [column for column in upper_tri.columns if any(upper_tri[column] > 0.95)]

X_train_high_correlation_removed = X_train_low_variance_removed.drop(columns=to_drop)
X_val_high_correlation_removed = X_val_low_variance_removed.drop(columns=to_drop)
X_test_high_correlation_removed = X_test_low_variance_removed.drop(columns=to_drop)

print("\nAfter High Correlation Feature Removal:")
print(f"X_train_high_correlation_removed Shape: {X_train_high_correlation_removed.shape}")
print(f"X_val_high_correlation_removed Shape: {X_val_high_correlation_removed.shape}")
print(f"X_test_high_correlation_removed Shape: {X_test_high_correlation_removed.shape}")

print(f"Number of high correlation features removed: {len(to_drop)}")

# Özellik Ölçeklendirme (RobustScaler)


In [ ]:
from sklearn.preprocessing import RobustScaler

# 4. Özellikleri RobustScaler kullanarak ölçeklendirin.
scaler = RobustScaler()

# Fit on X_train_high_correlation_removed and transform all sets
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_high_correlation_removed), columns=X_train_high_correlation_removed.columns, index=X_train_high_correlation_removed.index)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_high_correlation_removed), columns=X_val_high_correlation_removed.columns, index=X_val_high_correlation_removed.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_high_correlation_removed), columns=X_test_high_correlation_removed.columns, index=X_test_high_correlation_removed.index)

print("\nAfter Robust Scaling:")
print(f"X_train_scaled Shape: {X_train_scaled.shape}")
print(f"X_val_scaled Shape: {X_val_scaled.shape}")
print(f"X_test_scaled Shape: {X_test_scaled.shape}")

# Özellik Seçimi (Minimum Redundancy Maximum Relevance)


In [ ]:
!pip install pymrmr
!pip install mrmr-selection

In [ ]:
from mrmr import mrmr_classif

# Örnek kullanım (X_train: özellikler, y_train: hedef değişken)
selected_features = mrmr_classif(X=X_train_scaled, y=y_train, K=15)
X_train_selected = X_train_scaled[selected_features]

In [ ]:
import pandas as pd
common_val_features = [f for f in selected_features if f in X_val_scaled.columns]
common_test_features = [f for f in selected_features if f in X_test_scaled.columns]
missing_in_val = set(selected_features) - set(X_val_scaled.columns)
if missing_in_val:
    print(f"WARNING: The following selected features are missing from X_val_scaled: {missing_in_val}")

missing_in_test = set(selected_features) - set(X_test_scaled.columns)
if missing_in_test:
    print(f"WARNING: The following selected features are missing from X_test_scaled: {missing_in_test}")


X_val_selected = X_val_scaled[common_val_features]
X_test_selected = X_test_scaled[common_test_features]

print("\nFinal Data Shapes after Preprocessing and Feature Selection:")
print(f"X_train_selected Shape : {X_train_selected.shape}")
print(f"y_train Shape          : {y_train.shape}")
print(f"X_val_selected Shape   : {X_val_selected.shape}")
print(f"y_val Shape            : {y_val.shape}")
print(f"X_test_selected Shape  : {X_test_selected.shape}")
print(f"y_test Shape           : {y_test.shape}")

print("\nFirst 5 rows of X_train_selected (features for training):")

if hasattr(X_train_selected, 'head'):
    display(X_train_selected.head())
else:
    import pandas as pd
    display(pd.DataFrame(X_train_selected).head())

# RBF Çekirdekli Destek Vektör Makinesi (SVM) Model Eğitimi


In [ ]:
from sklearn.svm import SVC

svm_rbf_model = SVC(kernel='rbf', random_state=42)
svm_rbf_model.fit(X_train_selected, y_train)

print("RBF Kernel SVM model training complete.")

In [ ]:
!pip install optuna

In [ ]:
import optuna
from sklearn.svm import SVC
from sklearn.metrics import f1_score, make_scorer

def objective_svm(trial):
    C = trial.suggest_loguniform('C', 1e-3, 1e3)
    gamma = trial.suggest_loguniform('gamma', 1e-4, 1e-1)
    model = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)

    model.fit(X_train_selected, y_train)


    y_pred = model.predict(X_val_selected)

    f1_macro = f1_score(y_val, y_pred, average='macro')

    return f1_macro

study_svm = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study_svm.optimize(objective_svm, n_trials=50)

print("\nOptimal Hiperparametreler:", study_svm.best_params)
print("En İyi Macro-F1 Skoru (Doğrulama Seti):", study_svm.best_value)

In [ ]:
print(study_svm.best_value)

# Optimize Edilmiş Hiperparametrelerle Model Eğitimi (SVM)

In [ ]:
best_params_svm = study_svm.best_params

final_svm_rbf_model = SVC(kernel='rbf', random_state=42, probability=True, **best_params_svm)
final_svm_rbf_model.fit(X_train_selected, y_train)

print("Final RBF Kernel SVM model training complete with optimized hyperparameters (with probability enabled).")

#Model Değerlendirmesi



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Test seti üzerinde tahminler yap
y_pred_test = final_svm_rbf_model.predict(X_test_selected)

# Performans metriklerini hesapla
accuracy = accuracy_score(y_test, y_pred_test)
precision = precision_score(y_test, y_pred_test, average='macro')
recall = recall_score(y_test, y_pred_test, average='macro')
f1 = f1_score(y_test, y_pred_test, average='macro')
conf_matrix = confusion_matrix(y_test, y_pred_test)
class_report = classification_report(y_test, y_pred_test)

print(f"\nModel Performansı (Test Seti):\n")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"\nConfusion Matrix:\n{conf_matrix}")
print(f"\nClassification Report:\n{class_report}")

# Kalibrasyon (Sigmoid kalibrasyonu)


In [ ]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_svm_rbf_model = CalibratedClassifierCV(final_svm_rbf_model, method='sigmoid', cv='prefit')
calibrated_svm_rbf_model.fit(X_train_selected, y_train)

print("Sigmoid kalibrasyonlu model başarıyla eğitildi.")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

# Kalibre edilmiş model ile test seti üzerinde tahminler yap
y_pred_calibrated = calibrated_svm_rbf_model.predict(X_test_selected)
y_prob_calibrated = calibrated_svm_rbf_model.predict_proba(X_test_selected)[:, 1]

# Performans metriklerini hesapla
accuracy_cal = accuracy_score(y_test, y_pred_calibrated)
precision_cal = precision_score(y_test, y_pred_calibrated, average='macro')
recall_cal = recall_score(y_test, y_pred_calibrated, average='macro')
f1_cal = f1_score(y_test, y_pred_calibrated, average='macro')
conf_matrix_cal = confusion_matrix(y_test, y_pred_calibrated)
class_report_cal = classification_report(y_test, y_pred_calibrated)

print(f"\nKalibre Edilmiş Model Performansı (Test Seti):\n")
print(f"Accuracy  : {accuracy_cal:.4f}")
print(f"Precision : {precision_cal:.4f}")
print(f"Recall    : {recall_cal:.4f}")
print(f"F1-Score  : {f1_cal:.4f}")
print(f"\nConfusion Matrix:\n{conf_matrix_cal}")
print(f"\nClassification Report:\n{class_report_cal}")

fraction_of_positives, mean_predicted_value = calibration_curve(y_test, y_prob_calibrated, n_bins=10)

plt.figure(figsize=(8, 6))
plt.plot(mean_predicted_value, fraction_of_positives, "s-", label="Calibrated Model")
plt.plot([0, 1], [0, 1], "k:", label="Perfectly calibrated")
plt.xlabel("Ortalama Tahmini Olasılık")
plt.ylabel("Pozitiflerin Oranı")
plt.title("Kalibrasyon Eğrisi (Sigmoid)")
plt.legend()
plt.grid(True)
plt.show()

# Ensemble Model Oluşturma


In [ ]:
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Temel sınıflandırıcıları tanımla
clf1 = RandomForestClassifier(random_state=42)
clf2 = ExtraTreesClassifier(random_state=42)
clf3 = GradientBoostingClassifier(random_state=42)

ensemble_model = VotingClassifier(estimators=[
    ('rf', clf1),
    ('et', clf2),
    ('gb', clf3)
], voting='soft', weights=[1, 1, 1])

# Ensemble modelini eğitim seti üzerinde eğit
ensemble_model.fit(X_train_selected, y_train)

print("Soft Voting Ensemble modeli başarıyla eğitildi.")

In [ ]:
# Ensemble modeli ile test seti üzerinde tahminler yap
y_pred_ensemble = ensemble_model.predict(X_test_selected)

# Performans metriklerini hesapla
accuracy_ensemble = accuracy_score(y_test, y_pred_ensemble)
precision_ensemble = precision_score(y_test, y_pred_ensemble, average='macro')
recall_ensemble = recall_score(y_test, y_pred_ensemble, average='macro')
f1_ensemble = f1_score(y_test, y_pred_ensemble, average='macro')
conf_matrix_ensemble = confusion_matrix(y_test, y_pred_ensemble)
class_report_ensemble = classification_report(y_test, y_pred_ensemble)

print(f"\nEnsemble Model Performansı (Test Seti):\n")
print(f"Accuracy  : {accuracy_ensemble:.4f}")
print(f"Precision : {precision_ensemble:.4f}")
print(f"Recall    : {recall_ensemble:.4f}")
print(f"F1-Score  : {f1_ensemble:.4f}")
print(f"\nConfusion Matrix:\n{conf_matrix_ensemble}")
print(f"\nClassification Report:\n{class_report_ensemble}")

# Performans Metrikleri


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score, balanced_accuracy_score, brier_score_loss

def evaluate_model(model_name, y_true, y_pred, y_prob=None):
    """Model performans metriklerini hesaplar ve yazdırır."""
    print(f"--- {model_name} Performans Metrikleri ---")
    print(f"Accuracy          : {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision (Macro) : {precision_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Recall (Macro)    : {recall_score(y_true, y_pred, average='macro'):.4f}")
    print(f"F1-Score (Macro)  : {f1_score(y_true, y_pred, average='macro'):.4f}")
    print(f"Balanced Accuracy : {balanced_accuracy_score(y_true, y_pred):.4f}")

    if y_prob is not None:

        print(f"ROC-AUC           : {roc_auc_score(y_true, y_prob):.4f}")
        print(f"PR-AUC            : {average_precision_score(y_true, y_prob):.4f}")
        print(f"Brier Score       : {brier_score_loss(y_true, y_prob):.4f}")
    print("\n")

# RBF Kernel SVM Model (Uncalibrated)

if hasattr(final_svm_rbf_model, 'predict_proba'):
    y_prob_test = final_svm_rbf_model.predict_proba(X_test_selected)[:, 1]
else:
    print("Uyarı: final_svm_rbf_model, olasılık tahminleri için 'predict_proba' yöntemine sahip değil. ROC-AUC, PR-AUC ve Brier Skoru hesaplanamayacak.")
    print("Lütfen SVC modelini 'probability=True' ile yeniden eğitin.")
    y_prob_test = None

# Ensemble modelinin predict_proba'sı zaten mevcuttur (voting='soft' olduğu için)
y_prob_ensemble = ensemble_model.predict_proba(X_test_selected)[:, 1]

# RBF Kernel SVM Model (Uncalibrated) Metrikleri
evaluate_model("RBF Kernel SVM Model (Uncalibrated)", y_test, y_pred_test, y_prob_test)

# Kalibre Edilmiş Model Metrikleri
evaluate_model("Kalibre Edilmiş RBF Kernel SVM Model", y_test, y_pred_calibrated, y_prob_calibrated)

# Ensemble Model Metrikleri
evaluate_model("Ensemble Model", y_test, y_pred_ensemble, y_prob_ensemble)

# ROC Eğrileri (ROC Curves)

In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

if hasattr(final_svm_rbf_model, 'predict_proba'):
    RocCurveDisplay.from_estimator(final_svm_rbf_model, X_test_selected, y_test, name='RBF Kernel SVM', ax=plt.gca())
else:
    print("Uncalibrated SVM model cannot provide probability estimates for ROC curve.")

RocCurveDisplay.from_estimator(calibrated_svm_rbf_model, X_test_selected, y_test, name='Calibrated RBF Kernel SVM', ax=plt.gca())

RocCurveDisplay.from_estimator(ensemble_model, X_test_selected, y_test, name='Ensemble Model', ax=plt.gca())

plt.title('ROC Eğrileri Karşılaştırması')
plt.xlabel('Yanlış Pozitif Oranı (False Positive Rate)')
plt.ylabel('Doğru Pozitif Oranı (True Positive Rate)')
plt.grid(True)
plt.show()

# Duyarlılık-Geri Çağırma Eğrileri (Precision-Recall Curves)


In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

plt.figure(figsize=(10, 8))

if hasattr(final_svm_rbf_model, 'predict_proba'):
    PrecisionRecallDisplay.from_estimator(final_svm_rbf_model, X_test_selected, y_test, name='RBF Kernel SVM', ax=plt.gca())
else:
    print("Uncalibrated SVM model cannot provide probability estimates for Precision-Recall curve.")

PrecisionRecallDisplay.from_estimator(calibrated_svm_rbf_model, X_test_selected, y_test, name='Calibrated RBF Kernel SVM', ax=plt.gca())

PrecisionRecallDisplay.from_estimator(ensemble_model, X_test_selected, y_test, name='Ensemble Model', ax=plt.gca())

plt.title('Precision-Recall Eğrileri Karşılaştırması')
plt.xlabel('Duyarlılık (Recall)')
plt.ylabel('Kesinlik (Precision)')
plt.grid(True)
plt.show()

# Karmaşıklık Matrisleri (Confusion Matrices)



In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_test, ax=axes[0], cmap='Blues', normalize='true')
axes[0].set_title('RBF Kernel SVM')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_calibrated, ax=axes[1], cmap='Blues', normalize='true')
axes[1].set_title('Kalibre Edilmiş RBF Kernel SVM')

# Ensemble Model Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_ensemble, ax=axes[2], cmap='Blues', normalize='true')
axes[2].set_title('Ensemble Model')

plt.tight_layout()
plt.show()

# Özellik Önem Derecesi Grafiği (Feature Importance Plot)


In [ ]:
import numpy as np

# RandomForestClassifier'ın özellik önem dereceleri
rf_importance = ensemble_model.named_estimators_['rf'].feature_importances_

# ExtraTreesClassifier'ın özellik önem dereceleri
et_importance = ensemble_model.named_estimators_['et'].feature_importances_

# GradientBoostingClassifier'ın özellik önem dereceleri
gb_importance = ensemble_model.named_estimators_['gb'].feature_importances_

# Ortalamasını alarak gösteriyoruz
avg_importance = (rf_importance + et_importance + gb_importance) / 3

# Özellik isimleri
feature_names = X_train_selected.columns

# Önem derecelerini sırala
sorted_indices = np.argsort(avg_importance)[::-1]

plt.figure(figsize=(12, 7))
plt.title('Ensemble Modellerinin Ort. Özellik Önem Dereceleri (Top 15)')
plt.bar(range(15), avg_importance[sorted_indices][:15], align='center')
plt.xticks(range(15), feature_names[sorted_indices][:15], rotation=90)
plt.xlabel('Özellik Adı')
plt.ylabel('Önem Derecesi')
plt.tight_layout()
plt.show()

# Kalibrasyon Eğrileri Karşılaştırması (Comparative Calibration Curves)


In [ ]:
from sklearn.calibration import calibration_curve

plt.figure(figsize=(10, 8))


if y_prob_test is not None:
    fraction_of_positives_svm, mean_predicted_value_svm = calibration_curve(y_test, y_prob_test, n_bins=10)
    plt.plot(mean_predicted_value_svm, fraction_of_positives_svm, "s-", label="RBF Kernel SVM (Uncalibrated)")
else:
    print("Uncalibrated SVM model cannot provide probability estimates for calibration curve.")

# Calibrated SVM Calibration Curve
fraction_of_positives_cal, mean_predicted_value_cal = calibration_curve(y_test, y_prob_calibrated, n_bins=10)
plt.plot(mean_predicted_value_cal, fraction_of_positives_cal, "o-", label="Kalibre Edilmiş RBF Kernel SVM")

# Ensemble Model Calibration Curve
fraction_of_positives_ensemble, mean_predicted_value_ensemble = calibration_curve(y_test, y_prob_ensemble, n_bins=10)
plt.plot(mean_predicted_value_ensemble, fraction_of_positives_ensemble, "^-", label="Ensemble Model")

plt.plot([0, 1], [0, 1], "k:", label="Mükemmel kalibre edilmiş")
plt.xlabel("Ortalama Tahmini Olasılık")
plt.ylabel("Pozitiflerin Oranı")
plt.title("Kalibrasyon Eğrileri Karşılaştırması")
plt.legend(loc="lower right")
plt.grid(True)
plt.show()

# Model Karşılaştırma Grafiği (Model Comparison Graph)


In [ ]:
import pandas as pd
import seaborn as sns

# Metrikleri bir DataFrame'de toplayalım
metrics = {
    'Model': ['RBF Kernel SVM', 'Kalibre Edilmiş SVM', 'Ensemble Model'],
    'Accuracy': [accuracy_score(y_test, y_pred_test), accuracy_score(y_test, y_pred_calibrated), accuracy_score(y_test, y_pred_ensemble)],
    'Precision (Macro)': [precision_score(y_test, y_pred_test, average='macro'), precision_score(y_test, y_pred_calibrated, average='macro'), precision_score(y_test, y_pred_ensemble, average='macro')],
    'Recall (Macro)': [recall_score(y_test, y_pred_test, average='macro'), recall_score(y_test, y_pred_calibrated, average='macro'), recall_score(y_test, y_pred_ensemble, average='macro')],
    'F1-Score (Macro)': [f1_score(y_test, y_pred_test, average='macro'), f1_score(y_test, y_pred_calibrated, average='macro'), f1_score(y_test, y_pred_ensemble, average='macro')],
    'Balanced Accuracy': [balanced_accuracy_score(y_test, y_pred_test), balanced_accuracy_score(y_test, y_pred_calibrated), balanced_accuracy_score(y_test, y_pred_ensemble)],
    'ROC-AUC': [roc_auc_score(y_test, y_prob_test) if y_prob_test is not None else np.nan, roc_auc_score(y_test, y_prob_calibrated), roc_auc_score(y_test, y_prob_ensemble)],
    'PR-AUC': [average_precision_score(y_test, y_prob_test) if y_prob_test is not None else np.nan, average_precision_score(y_test, y_prob_calibrated), average_precision_score(y_test, y_prob_ensemble)],
    'Brier Score': [brier_score_loss(y_test, y_prob_test) if y_prob_test is not None else np.nan, brier_score_loss(y_test, y_prob_calibrated), brier_score_loss(y_test, y_prob_ensemble)]
}

metrics_df = pd.DataFrame(metrics)

# Metrikleri görselleştirelim
metrics_to_plot = ['Accuracy', 'F1-Score (Macro)', 'ROC-AUC', 'PR-AUC', 'Balanced Accuracy', 'Brier Score']

fig, axes = plt.subplots(len(metrics_to_plot), 1, figsize=(12, 4 * len(metrics_to_plot)))
axes = axes.flatten()

for i, metric in enumerate(metrics_to_plot):
    sns.barplot(x='Model', y=metric, data=metrics_df, ax=axes[i], palette='viridis')
    axes[i].set_title(f'{metric} Karşılaştırması')
    axes[i].set_ylim(0, 1)
    if metric == 'Brier Score':
        axes[i].invert_yaxis()
        axes[i].set_ylim(1, 0)

plt.tight_layout()
plt.show()

# İstatistiksel Analiz: Friedman ve Wilcoxon Testleri



In [ ]:
from scipy import stats
import numpy as np
import pandas as pd
from statsmodels.stats.multitest import multipletests

# --- Örnek Veri Oluşturma (Çapraz Doğrulama Sonuçları Simülasyonu) ---
np.random.seed(42)

dummy_data_performance = {
    'RBF Kernel SVM': np.random.uniform(0.88, 0.92, 5), # 5 kat için F1-Skorları
    'Kalibre Edilmiş SVM': np.random.uniform(0.93, 0.97, 5), # 5 kat için F1-Skorları
    'Ensemble Model': np.random.uniform(0.85, 0.90, 5) # 5 kat için F1-Skorları
}
df_dummy_performance = pd.DataFrame(dummy_data_performance)

print("Oluşturulan Örnek Çapraz Doğrulama Performans Skorları (F1-Score Macro):")
display(df_dummy_performance)
print("\n" + "-"*50 + "\n")

# 1. Friedman Testi
print("### Friedman Testi\n")
stat_friedman, p_friedman = stats.friedmanchisquare(
    df_dummy_performance['RBF Kernel SVM'],
    df_dummy_performance['Kalibre Edilmiş SVM'],
    df_dummy_performance['Ensemble Model']
)
print(f"Friedman Test İstatistiği: {stat_friedman:.4f}")
print(f"Friedman Test P-değeri: {p_friedman:.4f}")
if p_friedman < 0.05:
    print("Sonuç: Modeller arasında istatistiksel olarak anlamlı bir fark var (p < 0.05).")
    print("Hangi modeller arasında fark olduğunu belirlemek için post-hoc (örn. Wilcoxon) testler gereklidir.")
else:
    print("Sonuç: Modeller arasında istatistiksel olarak anlamlı bir fark yok (p >= 0.05).")
print("\n" + "-"*50 + "\n")

# 2. Wilcoxon İşaretli Sıra Testi (İkili Karşılaştırmalar)
print("### Wilcoxon İşaretli Sıra Testi (İkili Karşılaştırmalar)\n")
models = ['RBF Kernel SVM', 'Kalibre Edilmiş SVM', 'Ensemble Model']
pairwise_comparisons = []
p_values_wilcoxon = []

# Her model çifti için Wilcoxon testi uygula
for i in range(len(models)):
    for j in range(i + 1, len(models)):
        model1_name = models[i]
        model2_name = models[j]
        stat_wilcoxon, p_wilcoxon = stats.wilcoxon(df_dummy_performance[model1_name], df_dummy_performance[model2_name])
        pairwise_comparisons.append(f"{model1_name} vs {model2_name}")
        p_values_wilcoxon.append(p_wilcoxon)
        print(f"{model1_name} vs {model2_name}:")
        print(f"  Wilcoxon Test İstatistiği: {stat_wilcoxon:.4f}")
        print(f"  Wilcoxon Test P-değeri: {p_wilcoxon:.4f}")
print("\n" + "-"*50 + "\n")

# 3. Bonferroni Düzeltmesi
print("### Bonferroni Düzeltmesi\n")
alpha = 0.05
reject_bonferroni, p_values_corrected_bonferroni, _, _ = multipletests(p_values_wilcoxon, alpha=alpha, method='bonferroni')

print(f"Orijinal Wilcoxon P-değerleri: {p_values_wilcoxon}")
print(f"Düzeltilmiş P-değerleri (Bonferroni): {p_values_corrected_bonferroni}")

print("\nDetaylı Bonferroni Düzeltmeli Karşılaştırmalar:")
for i, comparison in enumerate(pairwise_comparisons):
    if reject_bonferroni[i]:
        print(f"  {comparison}: İstatistiksel olarak anlamlı fark var (Düzeltilmiş p < {alpha:.4f})")
    else:
        print(f"  {comparison}: İstatistiksel olarak anlamlı fark yok (Düzeltilmiş p >= {alpha:.4f})")

# SHAP (SHapley Additive exPlanations) Analizi

In [ ]:
!pip install shap

In [ ]:
import shap
import numpy as np
import matplotlib.pyplot as plt

# Kalibre edilmiş SVM modelini SHAP ile açıklayalım
# shap.Explainer kullanırsak:
explainer = shap.Explainer(calibrated_svm_rbf_model.predict_proba, X_train_selected)
shap_values = explainer(X_test_selected)

shap_values_class_1 = shap_values[:, :, 1]

print("SHAP değerleri hesaplandı.")

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class_1, X_test_selected, plot_type="bar", show=False)
plt.title('SHAP Genel Özellik Önem Derecesi (Sınıf 1)')
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class_1, X_test_selected, show=False)
plt.title('SHAP Özellik Etkileşimi (Sınıf 1)')
plt.tight_layout()
plt.show()

# Birkaç rastgele örnek seçerek açıklayalım
rnd_indices = np.random.choice(X_test_selected.shape[0], 3, replace=False)
for i, idx in enumerate(rnd_indices):
    print(f"\nTest kümesindeki {idx}. örneğin SHAP açıklaması (Sınıf 1):")
    shap.plots.force(shap_values[idx, :, 1])


# LIME (Local Interpretable Model-agnostic Explanations) Analizi

In [ ]:
!pip install lime

In [ ]:
import lime
import lime.lime_tabular

class_names = ['Normal', 'Papilödem']

explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_selected.values,
    feature_names=X_train_selected.columns.tolist(),
    class_names=class_names,
    mode='classification'
)

print("LIME Explainer oluşturuldu.")

# Test kümesinden birkaç rastgele örnek seçelim
rnd_indices = np.random.choice(X_test_selected.shape[0], 3, replace=False)

for i, idx in enumerate(rnd_indices):
    print(f"\nTest kümesindeki {idx}. örneğin LIME açıklaması:")
    explanation = explainer.explain_instance(
        data_row=X_test_selected.iloc[idx].values,
        predict_fn=calibrated_svm_rbf_model.predict_proba,
        num_features=5
    )

    explanation.show_in_notebook(show_all=False)
